# Covertype + TabFM + XAI
**Before running:** Runtime → Change runtime type → **T4 GPU**. Then Runtime → Run all.

## 1. Install

In [ ]:
!git clone -q --depth 1 https://github.com/google-research/tabfm.git
!pip install -q "./tabfm[pytorch]" lime scikit-learn scipy

## 2. The dataset
Each row is a 30m × 30m patch of forest in Colorado.
**Objective:** predict which of 7 tree types grows there (classification), using map data such as elevation, slope and distance to water.

In [ ]:
import pandas as pd
import sys
from pathlib import Path

for _directory in (Path.cwd(), Path.cwd() / "XAI" / "experiments", Path.cwd() / "experiments"):
    if (_directory / "controlled_lime_protocol.py").is_file():
        sys.path.insert(0, str(_directory))
        break
else:
    raise FileNotFoundError("Keep controlled_lime_protocol.py beside the experiment notebooks.")

from controlled_lime_protocol import CLASS_NAMES, load_controlled_covtype

X, y, X_train, X_test, y_train, y_test = load_controlled_covtype()
class_names = list(CLASS_NAMES)
df = X.assign(Cover_Type=y)

print("Shared sample shape:", df.shape)
print("Class counts:\n", y.value_counts().sort_index())
df.head()

## 3. Train TabFM and check performance
All model notebooks use the same seeded, stratified 10,000-row sample and 80/20 split from `controlled_lime_protocol.py`. Keep that helper beside the notebook (or upload it into the Colab runtime) before running.

In [ ]:
from sklearn.metrics import classification_report, accuracy_score

print("Shared modeling sample:", X.shape)
print("Train/test:", X_train.shape, X_test.shape)
print("Class counts:\n", y.value_counts().sort_index())

In [ ]:
import torch
from tabfm import TabFMClassifier
from tabfm import tabfm_v1_0_0_pytorch as tabfm_v1

tabfm_model = tabfm_v1.load(device="cuda", dtype=torch.float32)   # downloads ~6.5 GB the first time
model = TabFMClassifier(model=tabfm_model, n_estimators=8, cache_context=True)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, labels=list(range(7)), target_names=class_names, zero_division=0))

## 4. XAI 1: Permutation feature importance (global)
Shuffle one feature at a time and see how much accuracy drops. A bigger drop means the model relies on that feature more.

In [ ]:
from sklearn.inspection import permutation_importance
import matplotlib.pyplot as plt

result = permutation_importance(model, X_test[:500], y_test[:500], n_repeats=3, random_state=42)
importance = pd.Series(result.importances_mean, index=X.columns).sort_values()

importance.tail(10).plot.barh(title="Top 10 features (permutation importance)")
plt.xlabel("Drop in accuracy")
plt.show()

## 5. XAI 2: LIME (local)
Explain why the model made its prediction for one specific patch of forest.

In [ ]:
from lime.lime_tabular import LimeTabularExplainer

lime_background = X_train.sample(n=min(2000, len(X_train)), random_state=42)
categorical_indices = list(range(10, X.shape[1]))
explainer = LimeTabularExplainer(
    lime_background.values,
    feature_names=list(X.columns),
    class_names=class_names,
    categorical_features=categorical_indices,
    categorical_names={i: ["0", "1"] for i in categorical_indices},
    mode="classification",
    random_state=42,
)

def predict_fn(rows):
    return model.predict_proba(pd.DataFrame(rows, columns=X.columns))

row = X_test.iloc[0]
print("Actual:", y_test.iloc[0], "| Predicted:", model.predict(X_test.iloc[[0]])[0])

explanation = explainer.explain_instance(row.values, predict_fn, num_features=10, num_samples=1000, top_labels=1)
explanation.as_pyplot_figure(label=explanation.top_labels[0])
plt.show()

## Controlled LIME cost/quality comparison

The earlier single-row plot is illustrative only. The research comparison below uses shared test cases, LIME background, perturbation budgets, 10 reported explanation features, and repeat seeds. It also compares LIME perturbation dimensions of 18, 36, and 54 using the same nested feature subsets across models. Excluded features stay fixed at each case's observed values, so this tests LIME's local perturbation dimension—not model retraining with fewer predictors. The protocol is in `controlled_lime_protocol.py`. LIME's `exp.score` is a local surrogate fit score, not predictive accuracy. Runtime includes model calls made by LIME, but excludes explainer setup and warm-up.

In [ ]:
from IPython.display import display
from controlled_lime_protocol import benchmark_lime, evaluate_predictive_performance

model_name = "TabFM"
predict_proba_for_lime = lambda rows: model.predict_proba(pd.DataFrame(rows, columns=X.columns))
controlled_raw, controlled_summary = benchmark_lime(
    model, X_train, X_test, y_test, predict_proba_for_lime, model_name
)
print("Held-out predictive performance:", evaluate_predictive_performance(model, X_test, y_test))
display(controlled_summary.round(4))
controlled_raw.to_csv(f"{model_name.lower()}_controlled_lime_raw.csv", index=False)
controlled_summary.to_csv(f"{model_name.lower()}_controlled_lime_summary.csv", index=False)